# HMS — final statistics, tables and figures for the paper (CPU only), v3

v3 adds the extra training seeds of the original 200 s models (v10 seeds notebook) to Tables 2, 4 and 5, per-class truncation shifts, the window-shift seeds (v9s) in the asymmetry figure, the per-model prior-corrected hidden-test submissions, and one figure style for the paper.

**Inputs (Add Input → Your Work → Notebooks), Accelerator: None, Internet: off.** Attach the competition and every notebook output below (latest version of each):

| notebook | holds |
|---|---|
| HMS - Harmful Brain Activity Classification (competition) | labels, EEG lengths |
| `notebook86c832275b` (Version 5) | `results_v3`: 50 s / 200 s original models, seed 42 |
| `notebookd13836b6e5` | `shortcut`: reflect / truncate views of the original 200 s models |
| `notebookbb6726eb71` | `results_v5`: boundary-agnostic (BA) 200 s models, seed 42 |
| `notebook22e15f591d` (latest) | `results_v6`: BA no-reweight, seeds 42–3042 |
| `notebook92c60ba812` (latest) | `results_v7`: 50 s seeds 1042–4042 |
| `notebookdcb82b4ee1` (latest) | `results_v8`: CBraMod |
| v10 seeds notebook (latest) | `results_v10`: original 200 s models, seeds 1042/2042, with reflect/truncate views |
| v9s notebook (latest) | `results_v9`: window-shift runs, seeds 42–2042, and `v9_boundary_tests.csv` |

Nothing is trained. Every number is recomputed from saved out-of-fold predictions: calibration by cross-fitted temperature, uncertainty by one shared patient-level cluster bootstrap (the same resampled patients for every test, so differences between tests have valid intervals). Outputs: `/kaggle/working/paper/` (CSV per table, one Excel workbook, PNG+PDF figures).

In [1]:
CFG = dict(
    DATA_ROOT=None, OUT="/kaggle/working/paper", PRED_GLOBS=["/kaggle/input/**/*_valpred.npy", "/kaggle/input/**/*_oofpred.npy"],
    N_FOLDS=5, SEED=42, B=4000, T_GRID=[round(0.5 + 0.05 * i, 3) for i in range(51)], MIN_N=20,
    CTX_BINS=[-1, 10, 140, 1000], CTX_LABELS=["none (<10 s)", "partial (10-140 s)", "full (>=140 s)"],
    VOTE_GROUPS=[(1, 2, "1-2 votes"), (3, 9, "3-9 votes"), (10, 10_000, ">=10 votes")],
    SEEDS_50=[42, 1042, 2042, 3042, 4042], SEEDS_200=[42, 1042, 2042, 3042], SEEDS_ORIG=[42, 1042, 2042],
    V9_CSV_GLOB="/kaggle/input/**/v9_boundary_tests.csv",
)

In [2]:
import os, glob, time, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 300)
os.makedirs(CFG["OUT"], exist_ok=True)

def find_root():
    if CFG["DATA_ROOT"]:
        return CFG["DATA_ROOT"]
    for pat in ["/kaggle/input/*/train.csv", "/kaggle/input/*/*/train.csv", "/kaggle/input/*/*/*/train.csv"]:
        for h in glob.glob(pat):
            if os.path.isdir(os.path.join(os.path.dirname(h), "train_eegs")):
                return os.path.dirname(h)
    raise FileNotFoundError("Attach the competition data")
ROOT = find_root()
VOTES = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
CLASSES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]
P_COLS = [f"p_{c}" for c in CLASSES]
TABLES = {}
def save_table(name, d, show=True):
    TABLES[name] = d
    d.to_csv(f"{CFG['OUT']}/{name}.csv", index=False)
    if show:
        print(f"\n### {name}\n" + d.round(4).to_string(index=False))

## Labels, folds, out-of-fold rows, context availability

In [3]:
df = pd.read_csv(f"{ROOT}/train.csv")
df["n_votes"] = df[VOTES].sum(axis=1)
df[P_COLS] = df[VOTES].div(df["n_votes"], axis=0).to_numpy()
df["y"] = df[P_COLS].to_numpy().argmax(1)
df["center"] = df["eeg_label_offset_seconds"] + 25.0
sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
df["fold"] = -1
for k, (_, va) in enumerate(sgkf.split(df, df["y"], groups=df["patient_id"])):
    df.loc[df.index[va], "fold"] = k

def middle_row_per_eeg(rows):
    rows = rows.sort_values(["eeg_id", "eeg_label_offset_seconds"])
    g = rows.groupby("eeg_id")
    return rows[g.cumcount().to_numpy() == (g["eeg_id"].transform("size") // 2).to_numpy()]
VA = {f: middle_row_per_eeg(df[df.fold == f]).reset_index(drop=True) for f in range(CFG["N_FOLDS"])}
OOF = pd.concat([VA[f] for f in range(CFG["N_FOLDS"])], ignore_index=True)

t0 = time.time()
dur = {int(e): pq.ParquetFile(f"{ROOT}/train_eegs/{int(e)}.parquet").metadata.num_rows / 200.0 for e in OOF["eeg_id"].unique()}
print(f"read {len(dur)} EEG lengths in {time.time()-t0:.0f}s")
def extra_context(center, d, ctx=200):
    half = ctx / 2
    left = np.minimum(center, half); right = np.clip(np.minimum(d - center, half), 0, None)
    return left + right - np.minimum(center, 25) - np.clip(np.minimum(d - center, 25), 0, None)
OOF["dur_s"] = OOF["eeg_id"].map(dur)
OOF["extra_s"] = extra_context(OOF["center"].to_numpy(), OOF["dur_s"].to_numpy())
OOF["ctx_bin"] = pd.cut(OOF["extra_s"], CFG["CTX_BINS"], labels=CFG["CTX_LABELS"]).astype(str)
OOF["vote_group"] = ""
for lo, hi, name in CFG["VOTE_GROUPS"]:
    OOF.loc[(OOF.n_votes >= lo) & (OOF.n_votes <= hi), "vote_group"] = name

T = OOF[P_COLS].to_numpy(np.float64)
NV = OOF["n_votes"].to_numpy(); FOLD = OOF["fold"].to_numpy()
CB, VG = OOF["ctx_bin"].to_numpy(), OOF["vote_group"].to_numpy()
ALL = np.ones(len(OOF), bool); HV = NV >= 10
NONE, PART, FULL = (CB == CFG["CTX_LABELS"][0]), (CB == CFG["CTX_LABELS"][1]), (CB == CFG["CTX_LABELS"][2])
REAL = ~NONE
UP, INV = np.unique(OOF["patient_id"].to_numpy(), return_inverse=True)
print(f"{len(OOF)} EEGs, {len(UP)} patients")

read 17089 EEG lengths in 86s
17089 EEGs, 1950 patients


### Table 1 — data: context availability × annotation round, and label priors

In [4]:
ct = pd.crosstab(OOF["vote_group"], OOF["ctx_bin"], margins=True).reset_index()
save_table("T1a_context_by_votes", ct)
pri = {"all EEGs": ALL, "no extra context": NONE, "partial context": PART, "full context": FULL, ">=10 votes": HV, "no context & >=10 votes": NONE & HV}
rows = []
for k, m in pri.items():
    rows.append(dict(subset=k, n=int(m.sum()), median_votes=float(np.median(NV[m])), share_ge10=float(HV[m].mean()),
                     **{c: T[m, i].mean() for i, c in enumerate(CLASSES)}))
save_table("T1b_label_priors", pd.DataFrame(rows))


### T1a_context_by_votes
vote_group  full (>=140 s)  none (<10 s)  partial (10-140 s)   All
 1-2 votes              25          4526                 296  4847
 3-9 votes             297          2298                4047  6642
>=10 votes             169          2648                2783  5600
       All             491          9472                7126 17089

### T1b_label_priors
                 subset     n  median_votes  share_ge10  Seizure    LPD    GPD   LRDA   GRDA  Other
               all EEGs 17089           3.0      0.3277   0.1541 0.1425 0.1043 0.0654 0.1148 0.4189
       no extra context  9472           3.0      0.2796   0.0680 0.1360 0.1205 0.0453 0.0968 0.5334
        partial context  7126           3.0      0.3905   0.2631 0.1484 0.0814 0.0891 0.1308 0.2872
           full context   491           3.0      0.3442   0.2347 0.1808 0.1230 0.1099 0.2297 0.1219
             >=10 votes  5600          15.0      1.0000   0.0425 0.1829 0.1251 0.0740 0.0904 0.4851
no context & >=10

## Saved predictions

In [5]:
FILES = {}
for pat in CFG["PRED_GLOBS"]:
    for fp in glob.glob(pat, recursive=True):
        FILES.setdefault(os.path.basename(fp), fp)
print(f"{len(FILES)} prediction files found")

def pooled(fmt):
    """fmt contains {f}; returns pooled OOF predictions or None"""
    parts = []
    for f in range(CFG["N_FOLDS"]):
        fp = FILES.get(fmt.format(f=f))
        if fp is None:
            return None
        P = np.load(fp); assert len(P) == len(VA[f]), fp
        parts.append(P)
    return np.concatenate(parts).astype(np.float64)

def get(family, setting, seed=42, view="std"):
    """family: mamba/gru/transformer/cnn/mamba_causal/cbramod; setting: 50s1, 50s2, 200orig, 200ba, 200banr; view: std/reflect/truncate/trunc50"""
    sfx = "" if seed == 42 else f"_seed{seed}"
    if family == "cbramod":
        if seed != 42: return None
        if setting == "50s1" and view == "std": return pooled("cbramod_orig_ctx50_f{f}_standard_valpred.npy")
        if setting == "200orig": return pooled("cbramod_orig_ctx200_f{f}_" + {"std": "standard"}.get(view, view) + "_valpred.npy")
        if setting == "200banr": return pooled("cbramod_ba_noreweight_ctx200_f{f}_" + {"std": "ba"}.get(view, view) + "_valpred.npy")
        return None
    if setting in ("50s1", "50s2") and view == "std":
        st = setting[-2:]
        return pooled(f"{family}_ctx50_f{{f}}_{st}_valpred.npy") if seed == 42 else pooled(f"{family}_ctx50_f{{f}}{sfx}_{st}_valpred.npy")
    if setting == "200orig":
        if seed != 42:      # extra seeds of the original recipe (v10 seeds notebook)
            return pooled(f"{family}_ctx200_f{{f}}{sfx}_s1" + ("" if view == "std" else f"_{view}") + "_valpred.npy")
        if view == "std":
            return pooled(f"{family}_ctx200_f{{f}}_s1_valpred.npy")
        fp = FILES.get(f"{family}_ctx200_{view}_oofpred.npy")
        return np.load(fp).astype(np.float64) if fp else None
    if setting in ("200ba", "200banr"):
        var = "ba" if setting == "200ba" else "ba_noreweight"
        return pooled(f"{family}_{var}_ctx200_f{{f}}{sfx}_" + {"std": "ba"}.get(view, view) + "_valpred.npy")
    return None

def per_eeg_kl(P, eps=1e-15):
    return (T * (np.log(T + eps) - np.log(P + eps))).sum(1)
def temper(P, t):
    L = np.log(np.clip(P, 1e-12, 1)) / t; L -= L.max(1, keepdims=True); E = np.exp(L)
    return E / E.sum(1, keepdims=True)
def fit_temps(P, fit_mask):
    Ts = {}
    for f in range(CFG["N_FOLDS"]):
        tr = (FOLD != f) & fit_mask
        Ts[f] = min(CFG["T_GRID"], key=lambda t: (T[tr] * (np.log(T[tr] + 1e-15) - np.log(temper(P[tr], t) + 1e-15))).sum(1).mean())
    return Ts
def apply_temps(P, Ts):
    out = np.empty_like(P)
    for f, t in Ts.items():
        out[FOLD == f] = temper(P[FOLD == f], t)
    return out

_KL, _TS, _RAW = {}, {}, {}
def kl(family, setting, seed=42, view="std"):
    """calibrated per-EEG KL; intervention views reuse the temperatures fitted on the model's own standard view"""
    key = (family, setting, seed, view)
    if key in _KL:
        return _KL[key]
    P = get(family, setting, seed, view)
    if P is None:
        _KL[key] = None; return None
    base = (family, setting, seed)
    if base not in _TS:
        Pstd = P if view == "std" else get(family, setting, seed, "std")
        if Pstd is None:
            _KL[key] = None; return None
        _TS[base] = fit_temps(Pstd, HV if setting == "50s2" else ALL)
    _RAW[key] = per_eeg_kl(P)
    _KL[key] = per_eeg_kl(apply_temps(P, _TS[base]))
    return _KL[key]

# sanity: the shortcut-check standard view reproduces the saved original predictions
for fam in ["mamba", "gru", "transformer", "cnn"]:
    a, b = FILES.get(f"{fam}_ctx200_standard_oofpred.npy"), get(fam, "200orig")
    if a and b is not None:
        print(f"{fam} 200 s original: shortcut-check standard view vs saved predictions, max |dp| = {np.abs(np.load(a) - b).max():.4f}")

1560 prediction files found
mamba 200 s original: shortcut-check standard view vs saved predictions, max |dp| = 0.0010
gru 200 s original: shortcut-check standard view vs saved predictions, max |dp| = 0.0009
transformer 200 s original: shortcut-check standard view vs saved predictions, max |dp| = 0.0009
cnn 200 s original: shortcut-check standard view vs saved predictions, max |dp| = 0.0006


## Shared patient-level cluster bootstrap

In [6]:
W = np.random.default_rng(CFG["SEED"]).multinomial(len(UP), np.full(len(UP), 1.0 / len(UP)), size=CFG["B"]).astype(np.float32)
def rep_mean(d, msk):
    """bootstrap replicates of the mean of d over mask (patients resampled)"""
    s_ = np.bincount(INV[msk], weights=d[msk], minlength=len(UP)).astype(np.float32)
    n_ = np.bincount(INV[msk], minlength=len(UP)).astype(np.float32)
    return (W @ s_) / np.maximum(W @ n_, 1e-9)
def summarize(point, reps):
    reps = np.asarray(reps, float)
    if not np.isfinite(point) or not np.isfinite(reps).any():
        return dict(est=np.nan, ci_lo=np.nan, ci_hi=np.nan, p=np.nan)
    reps = reps[np.isfinite(reps)]
    lo, hi = np.percentile(reps, [2.5, 97.5])
    p = max(min(1.0, 2 * min((reps <= 0).mean(), (reps >= 0).mean())), 1.0 / CFG["B"])
    return dict(est=point, ci_lo=lo, ci_hi=hi, p=p)
def diff_test(ka, kb, msk):
    """mean(ka - kb) over mask; relative to mean(kb)"""
    d = ka - kb
    r = summarize(d[msk].mean(), rep_mean(d, msk))
    r["rel_pct"] = 100 * r["est"] / kb[msk].mean()
    r.update(n=int(msk.sum()), kl_a=ka[msk].mean(), kl_b=kb[msk].mean())
    return r
def rel_gain_reps(ka, kb, msk):
    return rep_mean(ka - kb, msk) / rep_mean(kb, msk), (ka - kb)[msk].mean() / kb[msk].mean()
def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = np.where(np.isfinite(p))[0]
    o = ok[np.argsort(p[ok])]; run = 0.0
    for i, ix in enumerate(o):
        run = max(run, min(1.0, (len(o) - i) * p[ix])); out[ix] = run
    return out

## Table 2 — architecture: calibrated pooled KL, mean ± SD over training seeds

In [7]:
FAMS = ["mamba", "gru", "cnn", "transformer", "cbramod"]
SETTINGS = [("50s1", CFG["SEEDS_50"], "50 s, stage 1"), ("50s2", CFG["SEEDS_50"], "50 s, stage 2 (>=10-vote fine-tune)"),
            ("200banr", CFG["SEEDS_200"], "200 s boundary-agnostic"), ("200orig", CFG["SEEDS_ORIG"], "200 s original (shortcut)")]
rows, SEEDS_HAVE = [], {}
for st, seeds, label in SETTINGS:
    for fam in FAMS:
        have = [s for s in seeds if kl(fam, st, s) is not None]
        SEEDS_HAVE[(fam, st)] = have
        if not have:
            continue
        a = np.array([kl(fam, st, s)[ALL].mean() for s in have]); h = np.array([kl(fam, st, s)[HV].mean() for s in have])
        raw = np.array([_RAW[(fam, st, s, "std")][ALL].mean() for s in have])
        Ts = [np.mean(list(_TS[(fam, st, s)].values())) for s in have]
        rows.append(dict(setting=label, model=fam, n_seeds=len(have), kl_all_mean=a.mean(), kl_all_sd=a.std(ddof=1) if len(a) > 1 else np.nan,
                         kl_ge10_mean=h.mean(), kl_ge10_sd=h.std(ddof=1) if len(h) > 1 else np.nan, kl_raw_all_mean=raw.mean(), temperature_mean=np.mean(Ts)))
save_table("T2_architecture_kl", pd.DataFrame(rows))


### T2_architecture_kl
                            setting       model  n_seeds  kl_all_mean  kl_all_sd  kl_ge10_mean  kl_ge10_sd  kl_raw_all_mean  temperature_mean
                      50 s, stage 1       mamba        5       0.6921     0.0097        0.5272      0.0083           0.7064            1.2080
                      50 s, stage 1         gru        5       0.7001     0.0049        0.5442      0.0046           0.7034            1.0980
                      50 s, stage 1         cnn        3       0.7186     0.0111        0.5619      0.0115           0.7232            1.1200
                      50 s, stage 1 transformer        3       0.7362     0.0086        0.5785      0.0068           0.7433            1.1433
                      50 s, stage 1     cbramod        1       0.7912        NaN        0.6367         NaN           0.8465            1.4200
50 s, stage 2 (>=10-vote fine-tune)       mamba        5       0.7611     0.0075        0.4130      0.0076           0.7779 

### Table 3 — pairwise architecture tests (per-EEG KL averaged over shared seeds)
Holm is reported over two families: every test in this table (`p_holm_all`), and the tests within one setting (`p_holm_setting`). Stage-2 models are scored on >=10-vote EEGs only.

In [8]:
rows = []
for st, seeds, label in SETTINGS[:3]:
    fams = [f for f in FAMS if SEEDS_HAVE.get((f, st))]
    for i, a in enumerate(fams):
        for b in fams[i + 1:]:
            common = [s for s in SEEDS_HAVE[(a, st)] if s in SEEDS_HAVE[(b, st)]]
            if not common:
                continue
            ka = np.mean([kl(a, st, s) for s in common], 0); kb = np.mean([kl(b, st, s) for s in common], 0)
            ka_wins = sum(kl(a, st, s)[HV if st == "50s2" else ALL].mean() < kl(b, st, s)[HV if st == "50s2" else ALL].mean() for s in common)
            for sname, msk in ([(">=10 votes", HV)] if st == "50s2" else [("all EEGs", ALL), (">=10 votes", HV)]):
                rows.append(dict(setting=label, comparison=f"{a} - {b}", seeds=len(common), subset=sname, first_better_in_seeds=f"{ka_wins}/{len(common)}",
                                 **diff_test(ka, kb, msk)))
pw = pd.DataFrame(rows)
pw["p_holm_all"] = holm(pw["p"].to_numpy())
pw["p_holm_setting"] = pw.groupby("setting")["p"].transform(lambda s: pd.Series(holm(s.to_numpy()), index=s.index))
save_table("T3_architecture_pairwise", pw)


### T3_architecture_pairwise
                            setting            comparison  seeds     subset first_better_in_seeds     est   ci_lo   ci_hi      p  rel_pct     n   kl_a   kl_b  p_holm_all  p_holm_setting
                      50 s, stage 1           mamba - gru      5   all EEGs                   4/5 -0.0080 -0.0227  0.0070 0.2745  -1.1456 17089 0.6921 0.7001      1.0000          0.3480
                      50 s, stage 1           mamba - gru      5 >=10 votes                   4/5 -0.0170 -0.0299 -0.0033 0.0225  -3.1255  5600 0.5272 0.5442      0.3150          0.1125
                      50 s, stage 1           mamba - cnn      3   all EEGs                   3/3 -0.0248 -0.0421 -0.0079 0.0065  -3.4475 17089 0.6938 0.7186      0.1170          0.0520
                      50 s, stage 1           mamba - cnn      3 >=10 votes                   3/3 -0.0328 -0.0560 -0.0119 0.0025  -5.8432  5600 0.5291 0.5619      0.0500          0.0250
                      50 s, stage 1   ma

## Table 4 — shortcut signatures of the original 200 s models (seed 42)
1. **No-context gain:** 200 s original vs the same family at 50 s, on EEGs whose 200 s window has no extra real signal. A model that reads only EEG cannot gain there.
2. **Reflect:** hiding the boundary by mirror-filling (all EEGs).
3. **Truncate vs 50 s model:** the 200 s model shown only the central 50 s, against the 50 s model, on full-context EEGs.
4. **Matched-input asymmetry:** on no-context EEGs and on truncated full-context EEGs the input has the same form; a model that reads only EEG should gain about equally on both. Asymmetry = relative gain (no-context) − relative gain (truncated full-context); 0 = no shortcut.

In [9]:
def asym_row(fam, label, k_none_a, k_none_b, k_full_a, k_full_b):
    rn, pn = rel_gain_reps(k_none_a, k_none_b, NONE)
    rf, pf = rel_gain_reps(k_full_a, k_full_b, FULL)
    r = summarize(pn - pf, rn - rf)
    return dict(model=fam, test=label, gain_none_pct=100 * pn, gain_full_trunc_pct=100 * pf,
                asymmetry_pct=100 * r["est"], ci_lo_pct=100 * r["ci_lo"], ci_hi_pct=100 * r["ci_hi"], p=r["p"])

rows, arows = [], []
for fam in FAMS:
    sets = []
    for s in CFG["SEEDS_ORIG"]:
        k50 = kl(fam, "50s1", s); ko = kl(fam, "200orig", s)
        kref, ktr = kl(fam, "200orig", s, "reflect"), kl(fam, "200orig", s, "truncate")
        if k50 is not None and ko is not None:
            sets.append((f"seed {s}", k50, ko, kref, ktr))
    full = [x for x in sets if x[3] is not None and x[4] is not None]
    if len(full) > 1:
        sets.append((f"mean of {len(full)} seeds",) + tuple(np.mean([x[i] for x in full], 0) for i in range(1, 5)))
    for lab, k50, korig, kref, ktr in sets:
        rows.append(dict(model=fam, seeds=lab, test="1 no-context gain (orig vs 50 s)", **diff_test(korig, k50, NONE)))
        if kref is not None:
            rows.append(dict(model=fam, seeds=lab, test="2 reflect vs standard (all EEGs)", **diff_test(kref, korig, ALL)))
        if ktr is not None:
            rows.append(dict(model=fam, seeds=lab, test="3 truncated orig vs 50 s (full context)", **diff_test(ktr, k50, FULL)))
            arows.append(asym_row(fam, f"original 200 s, {lab}", korig, k50, ktr, k50))
sig = pd.DataFrame(rows)
sig["p_holm"] = holm(sig["p"].to_numpy())                                   # over every row (per seed and seed means)
_has_mean = set(sig.loc[sig["seeds"].str.startswith("mean"), "model"])
_main = sig["seeds"].str.startswith("mean") | ~sig["model"].isin(_has_mean)
sig["p_holm_main"] = np.nan; sig.loc[_main, "p_holm_main"] = holm(sig.loc[_main, "p"].to_numpy())   # main family: seed means (single-seed models as is)
save_table("T4_shortcut_signatures", sig)


### T4_shortcut_signatures
      model           seeds                                    test     est   ci_lo   ci_hi      p  rel_pct     n   kl_a   kl_b  p_holm  p_holm_main
      mamba         seed 42        1 no-context gain (orig vs 50 s) -0.0677 -0.0903 -0.0452 0.0002 -10.6537  9472 0.5677 0.6354  0.0128          NaN
      mamba         seed 42        2 reflect vs standard (all EEGs)  0.2617  0.2226  0.3008 0.0002  41.2134 17089 0.8966 0.6349  0.0128          NaN
      mamba         seed 42 3 truncated orig vs 50 s (full context)  0.1396  0.0876  0.1981 0.0002  16.6005   491 0.9804 0.8408  0.0128          NaN
      mamba       seed 1042        1 no-context gain (orig vs 50 s) -0.0335 -0.0589 -0.0031 0.0315  -5.4655  9472 0.5790 0.6125  0.0315          NaN
      mamba       seed 1042        2 reflect vs standard (all EEGs)  0.3033  0.2617  0.3456 0.0002  47.2922 17089 0.9448 0.6414  0.0128          NaN
      mamba       seed 1042 3 truncated orig vs 50 s (full context)  0.2005  0

## Table 5 — boundary-agnostic models: residual-shortcut checks for every model and seed

In [10]:
rows = []
for fam in FAMS:
    for s in CFG["SEEDS_200"]:
        kba, ktr = kl(fam, "200banr", s), kl(fam, "200banr", s, "trunc50")
        k50 = kl(fam, "50s1", s) if kl(fam, "50s1", s) is not None else kl(fam, "50s1")
        if kba is None or ktr is None or k50 is None:
            continue
        r = asym_row(fam, f"BA seed {s}", kba, k50, ktr, k50)
        Pa, Pt = get(fam, "200banr", s), get(fam, "200banr", s, "trunc50")
        shift = (Pt[FULL] - Pa[FULL]).mean(0)
        r.update({f"shift_{c}": shift[i] for i, c in enumerate(CLASSES)}, max_abs_shift=np.abs(shift).max())
        arows.append(r)
asym = pd.DataFrame(arows)
asym["p_holm"] = holm(asym["p"].to_numpy())
save_table("T5_matched_input_asymmetry", asym)
rows = []
for fam in FAMS:
    for st, view, seeds in [("200orig", "truncate", CFG["SEEDS_ORIG"]), ("200banr", "trunc50", CFG["SEEDS_200"])]:
        for s in seeds:
            Pa, Pt = get(fam, st, s), get(fam, st, s, view)
            if Pa is None or Pt is None:
                continue
            sh = (Pt[FULL] - Pa[FULL]).mean(0)                         # raw probabilities, as in the earlier tables
            rows.append(dict(model=fam, setting=st, seed=s, **{c: sh[i] for i, c in enumerate(CLASSES)}))
save_table("T5b_truncation_shift_per_class", pd.DataFrame(rows))


### T5_matched_input_asymmetry
      model                            test  gain_none_pct  gain_full_trunc_pct  asymmetry_pct  ci_lo_pct  ci_hi_pct      p  shift_Seizure  shift_LPD  shift_GPD  shift_LRDA  shift_GRDA  shift_Other  max_abs_shift  p_holm
      mamba         original 200 s, seed 42       -10.6537              16.6005       -27.2541   -34.7042   -20.6640 0.0002            NaN        NaN        NaN         NaN         NaN          NaN            NaN  0.0072
      mamba       original 200 s, seed 1042        -5.4655              24.9139       -30.3794   -39.2039   -22.0285 0.0002            NaN        NaN        NaN         NaN         NaN          NaN            NaN  0.0072
      mamba       original 200 s, seed 2042        -9.7577              20.2696       -30.0273   -38.6630   -22.0774 0.0002            NaN        NaN        NaN         NaN         NaN          NaN            NaN  0.0072
      mamba original 200 s, mean of 3 seeds        -8.6663              20.5341     

## Table 6 — real long-context effect after the fix (BA view vs the same model limited to the central 50 s)
Per family, per-EEG KL averaged over seeds; Mamba-vs-other rows test whether Mamba gains more from context (interaction). CBraMod is reported but not interpretable (residual shortcut, Table 5).

In [11]:
rows, GAIN = [], {}
for fam in FAMS:
    seeds = [s for s in CFG["SEEDS_200"] if kl(fam, "200banr", s) is not None and kl(fam, "200banr", s, "trunc50") is not None]
    if not seeds:
        continue
    d = np.mean([kl(fam, "200banr", s) - kl(fam, "200banr", s, "trunc50") for s in seeds], 0)
    kb = np.mean([kl(fam, "200banr", s, "trunc50") for s in seeds], 0)
    GAIN[fam] = (d, kb)
    for sname, msk in [("EEGs with real context", REAL), ("full context", FULL), ("3-9 votes, with context", REAL & (VG == "3-9 votes")),
                       (">=10 votes, with context", REAL & HV), ("no context (sanity, should be 0)", NONE)]:
        if msk.sum() < CFG["MIN_N"]:
            continue
        r = summarize(d[msk].mean(), rep_mean(d, msk))
        rows.append(dict(model=fam, seeds=len(seeds), subset=sname, n=int(msk.sum()), diff=r["est"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"],
                         rel_pct=100 * r["est"] / kb[msk].mean(), p=r["p"]))
for other in ["gru", "transformer"]:
    if "mamba" in GAIN and other in GAIN:
        dd = GAIN["mamba"][0] - GAIN[other][0]
        r = summarize(dd[REAL].mean(), rep_mean(dd, REAL))
        rows.append(dict(model=f"interaction mamba - {other}", seeds=np.nan, subset="EEGs with real context", n=int(REAL.sum()),
                         diff=r["est"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"], rel_pct=np.nan, p=r["p"]))
ctxe = pd.DataFrame(rows)
ctxe["p_holm"] = holm(ctxe["p"].to_numpy())
save_table("T6_real_context_effect", ctxe)


### T6_real_context_effect
                          model  seeds                           subset    n    diff   ci_lo   ci_hi  rel_pct      p  p_holm
                          mamba    4.0           EEGs with real context 7617 -0.0239 -0.0297 -0.0182  -3.1386 0.0002  0.0055
                          mamba    4.0                     full context  491 -0.0395 -0.0628 -0.0170  -5.0552 0.0005  0.0065
                          mamba    4.0          3-9 votes, with context 4344 -0.0445 -0.0536 -0.0355  -4.8697 0.0002  0.0055
                          mamba    4.0         >=10 votes, with context 2952  0.0032 -0.0021  0.0096   0.6040 0.2525  1.0000
                          mamba    4.0 no context (sanity, should be 0) 9472 -0.0000 -0.0009  0.0008  -0.0022 0.9960  1.0000
                            gru    4.0           EEGs with real context 7617 -0.0192 -0.0240 -0.0145  -2.4245 0.0002  0.0055
                            gru    4.0                     full context  491 -0.0426 -0.0753 -0.0

## Table 7 — reweighting ablation (seed 42) and Table 8 — calibration

In [12]:
rows = []
for fam in ["mamba", "gru", "transformer"]:
    a, b = kl(fam, "200banr"), kl(fam, "200ba")
    if a is None or b is None:
        continue
    for sname, msk in [("all EEGs", ALL), (">=10 votes", HV)]:
        rows.append(dict(model=fam, subset=sname, comparison="no-reweight - reweight", **diff_test(a, b, msk)))
save_table("T7_reweighting", pd.DataFrame(rows))
rows = []
for (fam, st, s), Ts in sorted(_TS.items(), key=lambda x: (x[0][1], x[0][0], x[0][2])):
    raw, cal = _RAW.get((fam, st, s, "std")), _KL.get((fam, st, s, "std"))
    if raw is None or cal is None:
        continue
    m = HV if st == "50s2" else ALL
    rows.append(dict(model=fam, setting=st, seed=s, temperature=np.mean(list(Ts.values())), kl_raw=raw[m].mean(), kl_cal=cal[m].mean()))
save_table("T8_calibration", pd.DataFrame(rows), show=False)
print(TABLES["T8_calibration"].groupby(["setting", "model"])[["temperature", "kl_raw", "kl_cal"]].mean().round(3).to_string())


### T7_reweighting
      model     subset             comparison     est   ci_lo   ci_hi      p  rel_pct     n   kl_a   kl_b
      mamba   all EEGs no-reweight - reweight -0.0166 -0.0314 -0.0027 0.0185  -2.4050 17089 0.6727 0.6893
      mamba >=10 votes no-reweight - reweight -0.0323 -0.0465 -0.0186 0.0002  -5.8866  5600 0.5157 0.5479
        gru   all EEGs no-reweight - reweight -0.0353 -0.0495 -0.0213 0.0002  -4.9098 17089 0.6832 0.7184
        gru >=10 votes no-reweight - reweight -0.0531 -0.0690 -0.0372 0.0002  -9.1235  5600 0.5285 0.5815
transformer   all EEGs no-reweight - reweight -0.0260 -0.0382 -0.0140 0.0002  -3.4240 17089 0.7324 0.7584
transformer >=10 votes no-reweight - reweight -0.0372 -0.0533 -0.0215 0.0002  -6.0358  5600 0.5785 0.6157
                     temperature  kl_raw  kl_cal
setting model                                   
200ba   gru                1.060   0.720   0.718
        mamba              1.150   0.697   0.689
        transformer        1.090   0.761  

## Table 9 — Kaggle hidden test (private / public KL of the late submissions)
Entered from the Kaggle submissions page. `ge10` = predictions multiplied by π(≥10-vote training EEGs)/π(all training EEGs); `noctx` = by π(no-context EEGs)/π(all).

In [13]:
HT = pd.DataFrame([
    (2, "mamba", "200 BA", 42, "none", 0.592222, 0.495119), (5, "mamba", "200 orig", 42, "none", 0.542001, 0.446382),
    (6, "mamba", "50", 42, "none", 0.616707, 0.511592), (8, "gru", "200 BA", 42, "none", 0.626896, 0.515803),
    (9, "transformer", "200 BA", 42, "none", 0.635243, 0.526284), (10, "gru", "200 orig", 42, "none", 0.577058, 0.470861),
    (11, "transformer", "200 orig", 42, "none", 0.583351, 0.475581), (12, "gru", "50", 42, "none", 0.642315, 0.523472),
    (13, "transformer", "50", 42, "none", 0.648997, 0.533770), (14, "cnn", "50", 42, "none", 0.645963, 0.546354),
    (15, "constant", "prior all", np.nan, "none", 0.989693, 0.975832), (16, "constant", "prior noctx", np.nan, "none", 0.990905, 0.981253),
    (17, "constant", "prior ge10", np.nan, "none", 0.950651, 0.940303),
    (18, "mamba", "50", 42, "noctx", 0.551557, 0.455136), (19, "mamba", "200 BA", 42, "noctx", 0.528571, 0.442332),
    (20, "mamba", "50", 42, "ge10", 0.524216, 0.431618), (21, "mamba", "200 orig", 42, "ge10", 0.528213, 0.438629),
    (22, "mamba", "200 BA", 42, "ge10", 0.504660, 0.420012), (23, "gru", "50", 42, "ge10", 0.545799, 0.446865),
    (24, "transformer", "50", 42, "ge10", 0.547524, 0.456008), (25, "gru", "200 BA", 42, "ge10", 0.538079, 0.445479),
    (26, "transformer", "200 BA", 42, "ge10", 0.542675, 0.451757), (28, "gru", "200 orig", 42, "ge10", 0.554266, 0.456565),
    (29, "transformer", "200 orig", 42, "ge10", 0.586988, 0.490206), (32, "mamba", "200 BA", 1042, "ge10", 0.501420, 0.410949),
    (33, "mamba", "200 BA", 2042, "ge10", 0.491458, 0.413634), (34, "gru", "200 BA", 1042, "ge10", 0.537822, 0.438267),
    (35, "gru", "200 BA", 2042, "ge10", 0.544712, 0.446057)],
    columns=["version", "model", "setting", "seed", "prior_shift", "private", "public"])
save_table("T9a_hidden_test_all", HT, show=False)
two = HT[(HT.seed == 42) & HT.model.isin(["mamba", "gru", "transformer"])].pivot_table(index=["model", "prior_shift"], columns="setting", values="private")
two = two[["50", "200 orig", "200 BA"]].reset_index()
save_table("T9b_hidden_test_2x2", two)
sd = HT[(HT.setting == "200 BA") & (HT.prior_shift == "ge10")].groupby("model")["private"].agg(["count", "mean", "std", "min", "max"]).reset_index()
save_table("T9c_hidden_test_seeds", sd)


### T9b_hidden_test_2x2
      model prior_shift     50  200 orig  200 BA
        gru        ge10 0.5458    0.5543  0.5381
        gru        none 0.6423    0.5771  0.6269
      mamba        ge10 0.5242    0.5282  0.5047
      mamba       noctx 0.5516       NaN  0.5286
      mamba        none 0.6167    0.5420  0.5922
transformer        ge10 0.5475    0.5870  0.5427
transformer        none 0.6490    0.5834  0.6352

### T9c_hidden_test_seeds
      model  count   mean    std    min    max
        gru      3 0.5402 0.0039 0.5378 0.5447
      mamba      3 0.4992 0.0069 0.4915 0.5047
transformer      1 0.5427    NaN 0.5427 0.5427


## Figures

In [14]:
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.spines.top": False, "axes.spines.right": False,
                     "savefig.dpi": 300, "axes.titlesize": 8, "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7})
COL = {"50": "#9aa5b1", "200 orig": "#d1495b", "200 BA": "#2e86ab"}            # one palette for every figure
FAMC = {"mamba": "#2e86ab", "gru": "#5aa469", "cnn": "#9aa5b1", "transformer": "#f4a259", "cbramod": "#7b5ea7"}
NAME = {"mamba": "Mamba", "gru": "GRU", "cnn": "CNN", "transformer": "Transformer", "cbramod": "CBraMod"}
def savefig(fig, name):
    fig.tight_layout(); fig.savefig(f"{CFG['OUT']}/{name}.png", bbox_inches="tight"); fig.savefig(f"{CFG['OUT']}/{name}.pdf", bbox_inches="tight"); plt.close(fig)

# Fig: hidden test (seed 42), without correction, with the shared >=10-vote factor, with the per-model correction (if submitted)
panels = [("none", "No prior correction"), ("ge10", "Shared factor π(≥10)/π(all)")]
per_model = [p for p in HT["prior_shift"].unique() if p not in ("none", "ge10", "noctx")]
panels += [(p, f"Per-model correction ({p})") for p in per_model[:1]]
fig, axes = plt.subplots(1, len(panels), figsize=(7.2, 2.6), sharey=True)
for ax, (ps, title) in zip(np.atleast_1d(axes), panels):
    sub = HT[(HT.seed == 42) & (HT.prior_shift == ps)]
    for j, fam in enumerate(["mamba", "gru", "transformer"]):
        for i, st in enumerate(["50", "200 orig", "200 BA"]):
            v = sub[(sub.model == fam) & (sub.setting == st)]["private"]
            if len(v):
                ax.bar(j + (i - 1) * 0.27, v.iloc[0], 0.25, color=COL[st])
                ax.text(j + (i - 1) * 0.27, v.iloc[0] + 0.003, f"{v.iloc[0]:.3f}"[1:], ha="center", va="bottom", fontsize=5.5)
    ax.set_xticks(range(3)); ax.set_xticklabels(["Mamba", "GRU", "Transformer"]); ax.set_title(title)
    ax.set_ylim(0.45, 0.68)
np.atleast_1d(axes)[0].set_ylabel("Hidden-test KL (private)")
from matplotlib.patches import Patch
np.atleast_1d(axes)[-1].legend(handles=[Patch(color=COL[k], label=l) for k, l in [("50", "50 s"), ("200 orig", "200 s original"), ("200 BA", "200 s BA")]],
                               frameon=False, loc="upper right")
savefig(fig, "fig_hidden_test")

# Fig: matched-input asymmetry, forest plot. Circle + line = estimate with 95% patient-bootstrap interval (seed-averaged KL where
# several seeds exist); open diamonds = individual training seeds.
A5 = TABLES["T5_matched_input_asymmetry"]
rows_ = []   # (group, label, est, lo, hi, seed values, colour)
for fam in FAMS:
    sub = A5[(A5.model == fam) & A5.test.str.startswith("original 200 s")]
    if not len(sub):
        continue
    main = sub[sub.test.str.contains("mean of")]; main = main if len(main) else sub[sub.test.str.contains("seed 42")]
    seeds = sub[~sub.test.str.contains("mean of")]["asymmetry_pct"].to_numpy()
    r = main.iloc[0]; rows_.append(("Original 200 s", NAME[fam], r.asymmetry_pct, r.ci_lo_pct, r.ci_hi_pct, seeds if len(seeds) > 1 else [], FAMC[fam]))
for fam in FAMS:
    sub = A5[(A5.model == fam) & A5.test.str.startswith("BA seed")]
    if not len(sub):
        continue
    v = sub["asymmetry_pct"].to_numpy()
    if len(sub) == 1:
        r = sub.iloc[0]; rows_.append(("BA 200 s (mirror fill)", NAME[fam], r.asymmetry_pct, r.ci_lo_pct, r.ci_hi_pct, [], FAMC[fam]))
    else:
        rows_.append(("BA 200 s (mirror fill)", NAME[fam], v.mean(), np.nan, np.nan, v, FAMC[fam]))
v9f = sorted(glob.glob(CFG["V9_CSV_GLOB"], recursive=True))
if v9f:
    v9 = pd.read_csv(v9f[0]); v9 = v9[v9.test.str.startswith("edge probe")]
    vn = {"shift0": "no shift", "shift20_zero": "±20 s, zero fill", "shift20_mirror": "±20 s, mirror fill"}
    for fam in ["mamba", "gru"]:
        for var in ["shift0", "shift20_zero", "shift20_mirror"]:
            sub = v9[(v9.model == fam) & (v9.variant == var)]
            if not len(sub):
                continue
            main = sub[sub.seed.astype(str) == "mean"]; main = main if len(main) else sub[sub.seed.astype(str) == "42"]
            seeds = sub[sub.seed.astype(str) != "mean"]["asymmetry_pct"].to_numpy()
            r = main.iloc[0]; rows_.append(("50 s, edge probe", f"{NAME[fam]}, {vn[var]}", r.asymmetry_pct, r.ci_lo_pct, r.ci_hi_pct, seeds if len(seeds) > 1 else [], FAMC[fam]))
fig, ax = plt.subplots(figsize=(3.5, 0.17 * len(rows_) + 0.9))
y, yt, yl, prev = 0, [], [], None
for g, lab, est, lo, hi, seeds, c in rows_:
    if g != prev:
        y += 0.6; ax.text(-0.02, y, g, transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=7, fontweight="bold"); y += 1; prev = g
    if np.isfinite(lo):
        ax.plot([lo, hi], [y, y], color=c, lw=1.4)
    ax.plot(est, y, "o", color=c, ms=4)
    if len(seeds):
        ax.plot(seeds, np.full(len(seeds), y), "D", mfc="none", mec=c, ms=3, mew=0.8)
    yt.append(y); yl.append(lab); y += 1
ax.axvline(0, color="k", lw=0.6); ax.set_yticks(yt); ax.set_yticklabels(yl, fontsize=6.5); ax.invert_yaxis()
ax.set_xlabel("Asymmetry (percentage points)")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], color="k", marker="o", ms=4, lw=1.4, label="estimate, 95% interval"),
                   Line2D([], [], color="k", marker="D", mfc="none", ms=3, lw=0, label="individual seeds")], frameon=False, loc="upper center", bbox_to_anchor=(0.45, -0.16), ncol=2, fontsize=6)
savefig(fig, "fig_asymmetry")

# Fig: architecture, per-seed KL (only models that exist in each setting)
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4))
for ax, (st, msk, title) in zip(axes, [("50s1", ALL, "50 s, stage 1 (all)"), ("50s2", HV, "50 s, stage 2 (≥10 votes)"), ("200banr", ALL, "200 s BA (all)")]):
    fams = [f for f in ["mamba", "gru", "cnn", "transformer"] if SEEDS_HAVE.get((f, st))]
    for j, fam in enumerate(fams):
        vals = [kl(fam, st, s)[msk].mean() for s in SEEDS_HAVE[(fam, st)]]
        ax.scatter(np.full(len(vals), j) + np.linspace(-0.12, 0.12, len(vals)), vals, s=12, color=FAMC[fam], zorder=3)
        ax.hlines(np.mean(vals), j - 0.25, j + 0.25, color="k", lw=1)
    ax.set_xticks(range(len(fams))); ax.set_xticklabels([NAME[f] for f in fams]); ax.set_title(title)
axes[0].set_ylabel("Calibrated out-of-fold KL")
savefig(fig, "fig_architecture_seeds")

# Fig: truncation probability shift on full-context recordings (seed 42; raw probabilities)
T5b = TABLES.get("T5b_truncation_shift_per_class")
items = []
if T5b is not None and len(T5b):
    for fam, st, name, c in [("mamba", "200orig", "Mamba, original", "#d1495b"), ("cbramod", "200orig", "CBraMod, original", "#e8a0aa"),
                             ("mamba", "200banr", "Mamba, BA", "#2e86ab"), ("cbramod", "200banr", "CBraMod, BA", "#9cc9dd")]:
        r = T5b[(T5b.model == fam) & (T5b.setting == st) & (T5b.seed == 42)]
        if len(r):
            items.append((name, r[CLASSES].to_numpy()[0], c))
if items:
    fig, ax = plt.subplots(figsize=(3.5, 2.3))
    w = 0.8 / len(items)
    for i, (name, sh, c) in enumerate(items):
        ax.bar(np.arange(6) + (i - (len(items) - 1) / 2) * w, sh, w, label=name, color=c)
    ax.axhline(0, color="k", lw=0.6); ax.set_xticks(range(6)); ax.set_xticklabels(CLASSES)
    ax.set_ylabel("Mean probability change\n(central 50 s − full window)"); ax.legend(frameon=False, ncol=2, fontsize=6)
    savefig(fig, "fig_truncation_shift")

with pd.ExcelWriter(f"{CFG['OUT']}/hms_paper_tables.xlsx") as xw:
    for k, d in TABLES.items():
        d.to_excel(xw, sheet_name=k[:31], index=False)
print("\nwritten:", sorted(os.listdir(CFG["OUT"])))


written: ['T1a_context_by_votes.csv', 'T1b_label_priors.csv', 'T2_architecture_kl.csv', 'T3_architecture_pairwise.csv', 'T4_shortcut_signatures.csv', 'T5_matched_input_asymmetry.csv', 'T5b_truncation_shift_per_class.csv', 'T6_real_context_effect.csv', 'T7_reweighting.csv', 'T8_calibration.csv', 'T9a_hidden_test_all.csv', 'T9b_hidden_test_2x2.csv', 'T9c_hidden_test_seeds.csv', 'fig_architecture_seeds.pdf', 'fig_architecture_seeds.png', 'fig_asymmetry.pdf', 'fig_asymmetry.png', 'fig_hidden_test.pdf', 'fig_hidden_test.png', 'fig_truncation_shift.pdf', 'fig_truncation_shift.png', 'hms_paper_tables.xlsx']
